In [1]:
import pandas as pd
import psycopg2
from sklearn.model_selection import train_test_split
from dotenv import load_dotenv
import os



In [2]:
load_dotenv("/home/nikita/projects/first-project-yandex-mle/.env")

DB_DESTINATION_HOST = os.getenv("DB_DESTINATION_HOST")
DB_DESTINATION_PORT = os.getenv("DB_DESTINATION_PORT")
DB_DESTINATION_NAME = os.getenv("DB_DESTINATION_NAME")
DB_DESTINATION_USER = os.getenv("DB_DESTINATION_USER")
DB_DESTINATION_PASSWORD = os.getenv("DB_DESTINATION_PASSWORD")

In [3]:

conn = psycopg2.connect(
    host=DB_DESTINATION_HOST,
    port=DB_DESTINATION_PORT,
    dbname=DB_DESTINATION_NAME,
    user=DB_DESTINATION_USER,
    password=DB_DESTINATION_PASSWORD
)

sql = """
SELECT *
FROM cleaned_data_housing
"""

df = pd.read_sql(sql, conn)

conn.close()

df

/tmp/ipykernel_2883127/90319.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,id,flat_id,building_id,floor,kitchen_area,living_area,rooms,is_apartment,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
0,1,0,6220,9,9.90,19.900000,1,0,35.099998,9500000.0,1965,6,55.717113,37.781120,2.64,84,12,1
1,2,1,18012,7,0.00,16.600000,1,0,43.000000,13500000.0,2001,2,55.794849,37.608013,3.00,97,10,1
2,3,2,17821,9,9.00,32.000000,2,0,56.000000,13500000.0,2000,4,55.740040,37.761742,2.70,80,10,1
3,4,4,9293,3,3.00,14.000000,1,0,24.000000,5200000.0,1971,1,55.808807,37.707306,2.60,208,9,1
4,5,5,23964,9,0.00,0.000000,2,0,51.009998,8490104.0,2017,4,55.724728,37.743069,2.70,192,17,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97173,97174,141356,9503,8,6.00,42.000000,3,0,64.000000,10800000.0,1971,4,55.740402,37.834579,2.64,428,9,1
97174,97175,141358,3162,5,5.28,28.330000,2,0,41.110001,7400000.0,1960,1,55.727470,37.768677,2.48,80,5,0
97175,97176,141359,6513,7,5.30,20.000000,1,0,31.500000,9700000.0,1966,4,55.704315,37.506584,2.64,72,9,1
97176,97177,141360,23952,15,13.80,33.700001,2,0,65.300003,11750000.0,2017,4,55.699863,37.939564,2.70,480,25,1


In [4]:
df.drop(columns=['id', 'flat_id'], inplace=True) 


In [5]:
X_tr, X_val, y_tr, y_val = train_test_split(
    df,
    df['price']
)

In [6]:
# Тренировочная выборка
cat_features_tr = X_tr[["is_apartment", "building_type_int", "has_elevator"]]
potential_binary_features_tr = cat_features_tr.nunique() == 2

binary_cat_features_tr = cat_features_tr[potential_binary_features_tr[potential_binary_features_tr].index]
other_cat_features_tr = cat_features_tr[potential_binary_features_tr[~potential_binary_features_tr].index]
num_features_tr = X_tr.select_dtypes(['float'])

cat_features_val = X_val[["is_apartment", "building_type_int", "has_elevator"]]
potential_binary_features_val = cat_features_val.nunique() == 2

binary_cat_features_val = cat_features_val[potential_binary_features_val[potential_binary_features_val].index]
other_cat_features_val = cat_features_val[potential_binary_features_val[~potential_binary_features_val].index]
num_features_val = X_val.select_dtypes(['float'])

In [7]:
cat_features_tr["has_elevator"].value_counts()

has_elevator
1    65062
0     7821
Name: count, dtype: int64

In [9]:
df.describe()

,building_id,floor,kitchen_area,living_area,rooms,is_apartment,total_area,price,build_year,building_type_int,latitude,longitude,ceiling_height,flats_count,floors_total,has_elevator
count,97178.000000,97178.000000,97178.000000,97178.000000,97178.000000,97178.000000,97178.000000,9.717800e+04,97178.000000,97178.000000,97178.000000,97178.000000,97178.000000,97178.000000,97178.000000,97178.000000
mean,13140.837566,6.723096,8.082964,27.532788,1.942827,0.005248,51.470987,1.176918e+07,1984.048293,3.526755,55.728495,37.604187,2.686757,222.414487,12.998868,0.892692
std,6425.602609,4.501015,2.985974,14.325563,0.813515,0.072254,16.101386,4.467336e+06,19.321556,1.416711,0.103429,0.148815,0.130452,133.349654,5.114801,0.309506
min,256.000000,1.000000,0.000000,0.000000,1.000000,0.000000,11.000000,1.100000e+01,1914.000000,0.000000,55.427238,37.190639,2.450000,1.000000,1.000000,0.000000
25%,8129.250000,3.000000,6.000000,19.000000,1.000000,0.000000,38.299999,8.500000e+06,1969.000000,4.000000,55.648911,37.495985,2.640000,111.000000,9.000000,1.000000
50%,12782.000000,6.000000,8.400000,28.400000,2.000000,0.000000,48.000000,1.080000e+07,1979.000000,4.000000,55.717884,37.591732,2.640000,192.000000,12.000000,1.000000
75%,18495.000000,9.000000,10.000000,36.000000,3.000000,0.000000,60.299999,1.400000e+07,2002.000000,4.000000,55.814211,37.722549,2.700000,298.000000,17.000000,1.000000
max,24620.000000,20.000000,16.300000,74.000000,5.000000,1.000000,118.500000,2.840000e+07,2023.000000,6.000000,56.011032,37.946411,3.000000,638.000000,29.000000,1.000000


In [11]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor

# Если данные еще не загружены, раскомментируй:
# data = pd.read_csv("data/initial_data.csv")


# Убираем совсем странные значения цены
df = df[df["price"] > 100_000].copy()


target_col = "price"

feature_cols = [
    "building_id",
    "floor",
    "kitchen_area",
    "living_area",
    "rooms",
    "is_apartment",
    "total_area",
    "build_year",
    "building_type_int",
    "latitude",
    "longitude",
    "ceiling_height",
    "flats_count",
    "floors_total",
    "has_elevator",
]

X = df[feature_cols].copy()
y = df[target_col].copy()

# Логарифмируем таргет — для цены недвижимости это обычно работает лучше
y_log = np.log1p(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

model = CatBoostRegressor(
    iterations=10000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_state=42,
    verbose=100
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_test, y_test),
    use_best_model=True
)

# Предсказания в лог-шкале
y_pred_log = model.predict(X_test)

# Возвращаем в обычную шкалу цен
y_test_real = np.expm1(y_test)
y_pred_real = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
r2 = r2_score(y_test_real, y_pred_real)

print(f"MAE:  {mae:,.0f}")
print(f"RMSE: {rmse:,.0f}")
print(f"R2:   {r2:.4f}")

# Пример предсказаний
preds = pd.DataFrame({
    "y_true": y_test_real[:10].values,
    "y_pred": y_pred_real[:10]
})
preds

0:	learn: 0.3503113	test: 0.3507068	best: 0.3507068 (0)	total: 11.6ms	remaining: 1m 55s
100:	learn: 0.2057965	test: 0.2062260	best: 0.2062260 (100)	total: 740ms	remaining: 1m 12s
200:	learn: 0.1993784	test: 0.2011664	best: 0.2011664 (200)	total: 1.48s	remaining: 1m 12s
300:	learn: 0.1957623	test: 0.1987773	best: 0.1987773 (300)	total: 2.24s	remaining: 1m 12s
400:	learn: 0.1930928	test: 0.1974730	best: 0.1974730 (400)	total: 2.98s	remaining: 1m 11s
500:	learn: 0.1907245	test: 0.1965334	best: 0.1965334 (500)	total: 3.74s	remaining: 1m 10s
600:	learn: 0.1886414	test: 0.1958406	best: 0.1958406 (600)	total: 4.48s	remaining: 1m 10s
700:	learn: 0.1866787	test: 0.1952840	best: 0.1952840 (700)	total: 5.21s	remaining: 1m 9s
800:	learn: 0.1849047	test: 0.1948251	best: 0.1948251 (800)	total: 5.94s	remaining: 1m 8s
900:	learn: 0.1833063	test: 0.1945127	best: 0.1945127 (900)	total: 6.68s	remaining: 1m 7s
1000:	learn: 0.1817163	test: 0.1941603	best: 0.1941603 (1000)	total: 7.42s	remaining: 1m 6s
1100

,y_true,y_pred
0,15900000.0,1.540220e+07
1,12800000.0,1.535532e+07
2,12700000.0,1.390156e+07
3,9200000.0,8.798222e+06
4,8500000.0,9.205862e+06
5,12350000.0,1.064639e+07
6,8600000.0,7.864196e+06
7,7100000.0,7.147254e+06
8,6700000.0,8.657120e+06
9,9500000.0,1.071268e+07
